# System doboru modeli predykcyjnych szeregów czasowych (XGBoost + Meta-Learning)
**Cel:** Narzędzie do optymalizacji predykcji finansowych przy użyciu metod klasycznych i ML.

#### Zadanie 1: Wybór 6 modeli predykcyjnych szeregów czasowych
 W ramach projektu zidentyfikowaliśmy 6 kluczowych architektur, które zostaną zaimplementowane i porównane. Każdy model posiada specyficzne hiperparametry

#### Zadanie 2: Cel predykcji Dla chwili czasowej **t+1** przewidujemy:

1. **Główny cel:** Dokładną wartość zamknięcia (`Close`) – podejście regresyjne.
2. **Cel pomocniczy:** Kierunek zmiany (Wzrost/Spadek) w celu filtrowania transakcji przez Meta-Model.

---

## 1. Importy i Konfiguracja

In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import os

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, classification_report, precision_score

plt.style.use('seaborn-v0_8')
os.makedirs('stages', exist_ok=True)

## 2: Przygotowanie Danych 
(Feature Engineering)Pobieramy dane dla **Apple (AAPL)** oraz indeksu strachu **VIX** jako kontekstu rynkowego. Tworzymy cechy techniczne (RSI, SMA, ATR) oraz opóźnienia (Lags), które są niezbędne dla modeli nie-rekurencyjnych jak XGBoost.

### Pobieranie i Przetwarzanie Danych:

In [ ]:
print("Pobieranie danych...")
tickers = ['AAPL', '^VIX']
data = yf.download(tickers, start='2000-01-01', group_by='ticker', auto_adjust=False)

df = data['AAPL'].copy()
vix = data['^VIX']['Close'].copy()

df['RSI'] = df.ta.rsi(length=14)
df['SMA_20'] = df.ta.sma(length=20)
df['SMA_50'] = df.ta.sma(length=50)
df['ATR'] = df.ta.atr(length=14)
df['Dist_SMA'] = (df['Close'] - df['SMA_50']) / df['SMA_50']

df['VIX'] = vix
df['VIX'] = df['VIX'].ffill()
df['Panic_Mode'] = (df['VIX'] > 25).astype(int)
df['Rolling_Std'] = df['Close'].rolling(window=20).std()
df['VIX_Slope'] = df['VIX'].diff(5)

bb = df.ta.bbands(length=20, std=2)
df['BB_Width'] = bb['BBB_20_2.0_2.0']
df['RSI_Dist'] = abs(df['RSI'] - 50)

for lag in [1, 2, 3, 5]:
    df[f'Close_Lag_{lag}'] = df['Close'].shift(lag)

df['Target'] = df['Close'].shift(-1)
df.dropna(inplace=True)

print(f"Dane gotowe. Liczba wierszy: {len(df)}")
df.tail()

Base Model Accuracy: 44.73%


## 4: Implementacja Predyktorów
Porównamy dwa podejścia:
1. **Base Model (XGBoost):** Czysta regresja przewidująca cenę na kolejny dzień.
2. **Hybrid System (XGBoost + Meta-Labeling):** Regresja filtrowana przez klasyfikator Random Forest, który ocenia prawdopodobieństwo błędu modelu bazowego.

#### Krok A: Trening Modelu Bazowego (XGBoost)---

In [ ]:
feature_cols = [col for col in df.columns if col not in ['Target', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
X = df[feature_cols]
y = df['Target']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model_xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
model_xgb.fit(X_train, y_train)

results = X_test.copy()
results['Actual_Close'] = y_test
results['Predicted_Close'] = model_xgb.predict(X_test)
results['Prev_Close'] = results['Close_Lag_1']

mae = mean_absolute_error(results['Actual_Close'], results['Predicted_Close'])
print(f"MAE Modelu Bazowego: {mae:.2f}")

#### Krok B: Trening Meta-Modelu (Random Forest)
Meta-Model analizuje, kiedy XGBoost popełnia błędy kierunkowe. Jeśli Meta-Model przewiduje błąd, sygnał inwestycyjny jest blokowany.

Meta Labeling i Trening Filtra

In [ ]:
results['Signal'] = np.where(results['Predicted_Close'] > results['Prev_Close'], 1, -1)
results['Actual_Direction'] = np.where(results['Actual_Close'] > results['Prev_Close'], 1, -1)
results['Meta_Target'] = (results['Signal'] == results['Actual_Direction']).astype(int)

results['Model_Confidence'] = abs(results['Predicted_Close'] - results['Prev_Close']) / results['Prev_Close']

meta_features = ['RSI', 'ATR', 'Rolling_Std', 'Model_Confidence', 'VIX_Slope', 'BB_Width', 'RSI_Dist']
X_meta = results[meta_features]
y_meta = results['Meta_Target']

split_meta = int(len(results) * 0.5)
X_m_train, X_m_test = X_meta.iloc[:split_meta], X_meta.iloc[split_meta:]
y_m_train, y_m_test = y_meta.iloc[:split_meta], y_meta.iloc[split_meta:]

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [3, 5, 10],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, scoring='precision')
grid_search.fit(X_m_train, y_m_train)

meta_model = grid_search.best_estimator_
meta_preds = meta_model.predict(X_m_test)

print(f"Best Params: {grid_search.best_params_}")
print(classification_report(y_m_test, meta_preds))


## 5. Porównanie Wyników: Equity Curve
Symulacja strategii inwestycyjnej dla:
1. **Strategii Bazowej:** Podążaj za każdym sygnałem z XGBoost.
2. **Strategii Meta:** Podążaj za sygnałem tylko wtedy, gdy Meta-Model (Random Forest) potwierdzi jego wiarygodność.

In [ ]:
test_data = results.iloc[split_meta:].copy()
test_data['Meta_Filter'] = meta_preds

test_data['Return'] = (test_data['Actual_Close'] - test_data['Prev_Close']) / test_data['Prev_Close']
test_data['Strategy_Base'] = test_data['Return'] * test_data['Signal']
test_data['Strategy_Meta'] = test_data['Strategy_Base'] * test_data['Meta_Filter']

test_data['Equity_Base'] = (1 + test_data['Strategy_Base']).cumprod()
test_data['Equity_Meta'] = (1 + test_data['Strategy_Meta']).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(test_data['Equity_Base'], label='Base XGBoost Strategy', color='red', alpha=0.6)
plt.plot(test_data['Equity_Meta'], label='Meta-Filtered Strategy', color='green', linewidth=2)
plt.title('Porównanie Kapitału: Czysty XGBoost vs Meta-Filter')
plt.legend()
plt.grid(True)
plt.show()

## 6. Interpretowalność Modeli (SHAP)
Analiza, które cechy miały największy wpływ na decyzje Meta-Modelu (dlaczego odrzucił lub przyjął transakcję).

In [ ]:
explainer = shap.TreeExplainer(meta_model)
shap_values = explainer.shap_values(X_m_test)

if isinstance(shap_values, list):
    shap_vals_target = shap_values[1]
else:
    shap_vals_target = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_target, X_m_test, plot_type="bar")
plt.show()